In [ ]:
%pip install -q newspaper3k googlenewsdecoder
%pip install -q lxml_html_clean
import os
import pandas as pd
from google.colab import files
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from googlenewsdecoder import gnewsdecoder
from newspaper import Article, Config
print('Please upload your AIEDnews.csv file:')
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
config = Config()
config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
config.request_timeout = 10

def decode_google_news_url(source_url):
    try:
        decoded = gnewsdecoder(source_url, interval=0.2)
        if decoded.get('status'):
            return decoded.get('decoded_url')
    except Exception:
        pass
    return None

def fetch_full_text(target_url):
    if not target_url:
        return ''
    try:
        article = Article(target_url, config=config)
        article.download()
        article.parse()
        return article.text
    except Exception:
        return ''

def process_row(row):
    rss_link = row['link']
    real_url = decode_google_news_url(rss_link)
    full_text = fetch_full_text(real_url) if real_url else ''
    return {'title': row['title'], 'date': row['date'], 'month': row['month'], 'year': row['year'], 'outlet_domain': row['outlet_domain'], 'real_url': real_url, 'full_text': full_text}
results = []
print(f'\nExtracting full text for {len(df)} articles...')
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(process_row, row) for _, row in df.iterrows()]
    for future in tqdm(as_completed(futures), total=len(futures)):
        results.append(future.result())
output_df = pd.DataFrame(results)
output_path = 'AIEDnews_fulltext.csv'
output_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'\nExtraction complete! Downloading {output_path}...')
files.download(output_path)


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import json
import re
import time
INPUT_FILE = 'AIEDnews_fulltext_round2_completed.csv'
OUTPUT_FILE = 'AIED.csv'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36', 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8', 'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7', 'Referer': 'https://www.google.com/'}

def extract_from_json_ld(soup):
    scripts = soup.find_all('script', type='application/ld+json')
    for script in scripts:
        if not script.string:
            continue
        try:
            data = json.loads(script.string)
            items = data if isinstance(data, list) else [data]
            for item in items:
                if item.get('@type') in ['NewsArticle', 'Article']:
                    body = item.get('articleBody')
                    if body and len(body) > 100:
                        return body
        except Exception:
            continue
    return None

def extract_content(url, domain):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=12)
        if resp.status_code != 200:
            return (None, 0)
        soup = BeautifulSoup(resp.content, 'html.parser')
        json_body = extract_from_json_ld(soup)
        if json_body:
            return (json_body.strip(), len(json_body.strip()))
        body = None
        if 'dantri.com.vn' in str(domain):
            body = soup.find('div', class_='singular-content') or soup.find('div', class_='dt-news__content') or soup.find('article')
        elif 'laodong.vn' in str(domain):
            body = soup.find('div', class_='article-content') or soup.find('div', class_='art-body') or soup.find('div', class_='content-wrapper')
        elif 'vov.vn' in str(domain):
            body = soup.find('div', class_='article-content') or soup.find('div', class_='row text-long')
        elif 'vtv.vn' in str(domain):
            body = soup.find('div', class_='ta-content') or soup.find('div', class_='noidung')
        if not body:
            body = soup.find('article') or soup
        paragraphs = [p.get_text(strip=True) for p in body.find_all(['p', 'div', 'span']) if len(p.get_text(strip=True)) > 30]
        seen = set()
        unique_paragraphs = []
        for p in paragraphs:
            if p not in seen:
                seen.add(p)
                unique_paragraphs.append(p)
        clean_text = '\n\n'.join(unique_paragraphs)
        return (clean_text, len(clean_text)) if len(clean_text) > 100 else (None, 0)
    except Exception:
        return (None, 0)
df = pd.read_csv(INPUT_FILE)
missing_mask = df['full_text'].isna() | (df['char_count'] < 100)
missing_indices = df[missing_mask].index
print(f'Targeting {len(missing_indices)} articles for Round 2 extraction...')
recovered = 0
for i, idx in enumerate(missing_indices):
    url = df.loc[idx, 'real_url']
    domain = df.loc[idx, 'outlet_domain']
    text, length = extract_content(url, domain)
    if text:
        df.loc[idx, 'full_text'] = text
        df.loc[idx, 'char_count'] = length
        recovered += 1
    if (i + 1) % 20 == 0 or i + 1 == len(missing_indices):
        print(f'Processed {i + 1}/{len(missing_indices)} | Recovered: {recovered}')
    time.sleep(0.3)
print(f'\n--- Round 2 Finished ---')
print(f'Successfully recovered {recovered} / {len(missing_indices)} articles.')
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
from google.colab import files
files.download(OUTPUT_FILE)


In [ ]:
%pip install cloudscraper
import pandas as pd
import requests
import cloudscraper
from bs4 import BeautifulSoup
import json
import time
from google.colab import files
FILE_NAME = 'AIEDnews_fulltext_round2_completed.csv'
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36', 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7', 'Referer': 'https://www.google.com/'}
scraper = cloudscraper.create_scraper()

def extract_from_json_ld(soup):
    scripts = soup.find_all('script', type='application/ld+json')
    for script in scripts:
        if not script.string:
            continue
        try:
            data = json.loads(script.string)
            items = data if isinstance(data, list) else [data]
            for item in items:
                if item.get('@type') in ['NewsArticle', 'Article']:
                    body = item.get('articleBody')
                    if body and len(body) > 100:
                        return body
        except Exception:
            continue
    return None

def fetch_html(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=8)
        if resp.status_code == 200:
            return resp.content
    except Exception:
        pass
    try:
        resp = scraper.get(url, headers=HEADERS, timeout=12)
        if resp.status_code == 200:
            return resp.content
    except Exception:
        pass
    return None

def extract_content(url, domain):
    html = fetch_html(url)
    if not html:
        return (None, 0)
    try:
        soup = BeautifulSoup(html, 'html.parser')
        json_body = extract_from_json_ld(soup)
        if json_body:
            return (json_body.strip(), len(json_body.strip()))
        body = None
        if 'dantri.com.vn' in str(domain):
            body = soup.find('div', class_='singular-content') or soup.find('div', class_='dt-news__content') or soup.find('article')
        elif 'laodong.vn' in str(domain):
            body = soup.find('div', class_='article-content') or soup.find('div', class_='art-body') or soup.find('div', class_='content-wrapper')
        elif 'vov.vn' in str(domain):
            body = soup.find('div', class_='article-content') or soup.find('div', class_='row text-long')
        elif 'vtv.vn' in str(domain):
            body = soup.find('div', class_='ta-content') or soup.find('div', class_='noidung')
        if not body:
            body = soup.find('article') or soup
        paragraphs = [p.get_text(strip=True) for p in body.find_all(['p', 'div', 'span']) if len(p.get_text(strip=True)) > 30]
        seen = set()
        unique_paragraphs = []
        for p in paragraphs:
            if p not in seen:
                seen.add(p)
                unique_paragraphs.append(p)
        clean_text = '\n\n'.join(unique_paragraphs)
        return (clean_text, len(clean_text)) if len(clean_text) > 100 else (None, 0)
    except Exception:
        return (None, 0)
df = pd.read_csv(FILE_NAME)
missing_mask = df['full_text'].isna() | (df['char_count'] < 100)
missing_indices = df[missing_mask].index
print(f'Remaining articles to recover: {len(missing_indices)} / {len(df)}')
if len(missing_indices) == 0:
    print('Dataset is 100% complete!')
else:
    recovered_this_run = 0
    for i, idx in enumerate(missing_indices):
        url = df.loc[idx, 'real_url']
        domain = df.loc[idx, 'outlet_domain']
        text, length = extract_content(url, domain)
        if text:
            df.loc[idx, 'full_text'] = text
            df.loc[idx, 'char_count'] = length
            recovered_this_run += 1
        if (i + 1) % 10 == 0 or i + 1 == len(missing_indices):
            df.to_csv(FILE_NAME, index=False, encoding='utf-8-sig')
            print(f'Progress: {i + 1}/{len(missing_indices)} | Total Recovered This Pass: {recovered_this_run}')
        time.sleep(0.5)
    print(f'\nFinished pass. Recovered {recovered_this_run} new articles.')
    df.to_csv(FILE_NAME, index=False, encoding='utf-8-sig')
    files.download(FILE_NAME)


In [ ]:
%pip install selenium webdriver-manager
!apt-get update
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable
import os
import time
import json
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
FILE_NAME = 'AIEDnews_fulltext_round2_completed.csv'
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36')
chrome_options.binary_location = '/opt/google/chrome/google-chrome'
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-features=NetworkService')
chrome_options.add_argument('--no-default-browser-check')
chrome_options.add_argument('--disable-extensions')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_argument('--disable-logging')
chrome_options.add_argument('--log-level=3')
chrome_options.add_argument('--silent')
chrome_options.add_argument('--remote-debugging-port=9222')
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

def extract_from_json_ld(soup):
    scripts = soup.find_all('script', type='application/ld+json')
    for script in scripts:
        if not script.string:
            continue
        try:
            data = json.loads(script.string)
            items = data if isinstance(data, list) else [data]
            for item in items:
                if item.get('@type') in ['NewsArticle', 'Article']:
                    body = item.get('articleBody')
                    if body and len(body) > 100:
                        return body
        except Exception:
            continue
    return None

def extract_content_selenium(url, domain):
    try:
        driver.get(url)
        time.sleep(2.5)
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, 'html.parser')
        json_body = extract_from_json_ld(soup)
        if json_body:
            return (json_body.strip(), len(json_body.strip()))
        body = None
        if 'dantri.com.vn' in str(domain):
            body = soup.find('div', class_='singular-content') or soup.find('div', class_='dt-news__content') or soup.find('article')
        elif 'laodong.vn' in str(domain):
            body = soup.find('div', class_='article-content') or soup.find('div', class_='art-body') or soup.find('div', class_='content-wrapper')
        elif 'vov.vn' in str(domain):
            body = soup.find('div', class_='article-content') or soup.find('div', class_='row text-long')
        elif 'vtv.vn' in str(domain):
            body = soup.find('div', class_='ta-content') or soup.find('div', class_='noidung')
        if not body:
            body = soup.find('article') or soup
        paragraphs = [p.get_text(strip=True) for p in body.find_all(['p', 'div', 'span']) if len(p.get_text(strip=True)) > 30]
        seen = set()
        unique_paragraphs = []
        for p in paragraphs:
            if p not in seen:
                seen.add(p)
                unique_paragraphs.append(p)
        clean_text = '\n\n'.join(unique_paragraphs)
        return (clean_text, len(clean_text)) if len(clean_text) > 100 else (None, 0)
    except Exception:
        return (None, 0)
df = pd.read_csv(FILE_NAME)
missing_mask = df['full_text'].isna() | (df['char_count'] < 100)
missing_indices = df[missing_mask].index
print(f'Targeting {len(missing_indices)} stubborn articles with Selenium...')
recovered_this_run = 0
try:
    for i, idx in enumerate(missing_indices):
        url = df.loc[idx, 'real_url']
        domain = df.loc[idx, 'outlet_domain']
        text, length = extract_content_selenium(url, domain)
        if text:
            df.loc[idx, 'full_text'] = text
            df.loc[idx, 'char_count'] = length
            recovered_this_run += 1
        if (i + 1) % 10 == 0 or i + 1 == len(missing_indices):
            df.to_csv(FILE_NAME, index=False, encoding='utf-8-sig')
            print(f'Progress: {i + 1}/{len(missing_indices)} | Recovered: {recovered_this_run}')
finally:
    driver.quit()
print(f'\nFinished pass. Successfully recovered {recovered_this_run} articles.')
df.to_csv(FILE_NAME, index=False, encoding='utf-8-sig')
files.download(FILE_NAME)
